In [ ]:
import pandas as pd
import gc
import numpy as np

In [ ]:
df3 = pd.read_pickle('../data/df3.pkl')

### Information on the Patron Categories: 

- Student (Primary 1 to JC/Poly) --> 7 - 19 years old
- Adult --> 20 - 59 years old
- Senior Citizen --> 60 years and above

Average Walking Speeds by Age:
- Children (6-12 years): Around 3.0 to 4.5 km/h (1.9 to 2.8 mph)
- Adolescents (13-19 years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)
- Adults (20-59 years): Around 5.0 to 6.0 km/h (3.1 to 3.7 mph)
- Older Adults (60+ years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)

To calculate the walking speed of each group, I take the midpoint of the range. 
For Children and Adolescents, I took the midpoint of both ranges and took the average.

In [ ]:
# We first filter for 3 relevant PATRON_CATG_ID_NUM.
patron_map = {1: 'Adult', 3: 'Student', 4: 'Senior Citizen'}
df3['PATRON_CATG_DESC_TXT'] = df3['PATRON_CATG_ID_NUM'].map(patron_map)

walking_speed_ms = {
    'Student': 4.125 / 3.6,      # (3.75 + 4.5) / 2 = 4.125 km/h → ~1.146 m/s
    'Adult': 1.08,          
    'Senior Citizen': 0.80  
}

df3['walking_speed_ms'] = df3['PATRON_CATG_DESC_TXT'].map(walking_speed_ms)

# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: df3 is already sorted by CRD_NUM and ENTRY_TM. Flag last row of each CRD_NUM
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


#### Day starts at 5am

In [ ]:
df3['service_day'] = (df3['ENTRY_TM'] - pd.Timedelta(hours=5)).dt.date
target_day = pd.Timestamp('2025-02-12').date()

df3 = df3[df3['service_day'] == target_day].reset_index(drop=True)

In [ ]:
df3 = df3.sort_values(["CRD_NUM", "ENTRY_TM"]).reset_index(drop=True)

In [ ]:
df3["next_ENTRY_TM"] = df3.groupby("CRD_NUM")["ENTRY_TM"].shift(-1)
df3["next_ORIG_LOC_ID_NUM"] = df3.groupby("CRD_NUM")["ORIG_LOC_ID_NUM"].shift(-1)
df3["next_BUS_SVC_NUM"] = df3.groupby("CRD_NUM")["BUS_SVC_NUM"].shift(-1)
df3["next_TRNSPT_MODE_CD"] = df3.groupby("CRD_NUM")["TRNSPT_MODE_CD"].shift(-1)

In [ ]:
# Whether this is the last journey stage of the day.
df3["is_last_stage"] = df3["next_ENTRY_TM"].isna()

# Missing critical informations. 
df3['missing_info'] = (
    df3['ENTRY_TM'].isna() |
    df3['EXIT_TM'].isna() |
    df3['ORIG_LATITUDE'].isna() |
    df3['ORIG_LONGITUDE'].isna() |
    df3['DEST_LATITUDE'].isna() |
    df3['DEST_LONGITUDE'].isna() |
    df3["ORIG_LOC_ID_NUM"].isna() |
    df3["DEST_LOC_ID_NUM"].isna()
)

In [ ]:
# flags consecutive rides on the same bus service number/same station
df3["same_bus_service"] = (
    df3["BUS_SVC_NUM"].notna() &
    df3["next_BUS_SVC_NUM"].notna() &
    (df3["BUS_SVC_NUM"] == df3["next_BUS_SVC_NUM"])
)
df3["same_station_consecutive"] = (
    df3["DEST_STATION_NAME"].notna() &
    df3["next_orig_station"].notna() &
    (df3["DEST_STATION_NAME"] == df3["next_orig_station"])
)

In [ ]:
df3["return_or_intermediate"] = (
    df3["same_bus_service"] |
    (
        (df3["TRNSPT_MODE_CD"] == 2) &
        (df3["next_TRNSPT_MODE_CD"] == 2) &
        df3["same_station_consecutive"]
    )
)

In [ ]:
df3["binary_flag"] = (
    df3["is_last_stage"] |
    df3["missing_info"] |
    df3["return_or_intermediate"]
)

In [ ]:
df3["binary_flag"].value_counts()


In [ ]:
# Summary counts
print("Unique commuters:", df3["CRD_NUM"].nunique())
print("Last stages:", df3["is_last_stage"].sum())

print("\nMissing info:")
print(df3["missing_info"].value_counts(dropna=False))

print("\nSame bus service:")
print(df3["same_bus_service"].value_counts(dropna=False))

print("\nSame station consecutive:")
print(df3["same_station_consecutive"].value_counts(dropna=False))

print("\nReturn / intermediate:")
print(df3["return_or_intermediate"].value_counts(dropna=False))

print("\nbinary_flag:")
print(df3["binary_flag"].value_counts(dropna=False))

In [ ]:
# Reasons why flagged.
df3["binary_flag_reason"] = np.select(
    [
        df3["is_last_stage"],
        df3["missing_info"],
        df3["return_or_intermediate"]
    ],
    [
        "last_stage",
        "missing_info",
        "return_or_intermediate"
    ],
    default="candidate_transfer"
)

In [ ]:
print(df3["binary_flag_reason"].value_counts(dropna=False))

In [ ]:
# -----------------------------------
# Binary-only journey sequence
# -----------------------------------
df3["binary_full_journey_seq"] = (
    df3.groupby("CRD_NUM")["binary_flag"]
       .shift(fill_value=False)
       .groupby(df3["CRD_NUM"])
       .cumsum()
)

binary_journey_first = (
    df3.drop_duplicates(subset=["CRD_NUM", "binary_full_journey_seq"], keep="first")
       [["CRD_NUM", "binary_full_journey_seq", "ORIG_STATION_NAME", "ENTRY_TM"]]
       .rename(columns={
           "ORIG_STATION_NAME": "(BINARY_Only)_journey_orig_station",
           "ENTRY_TM": "(BINARY_Only)_journey_entry_tm"
       })
)

binary_journey_last = (
    df3.drop_duplicates(subset=["CRD_NUM", "binary_full_journey_seq"], keep="last")
       [["CRD_NUM", "binary_full_journey_seq",
         "DEST_STATION_NAME", "EXIT_TM",
         "next_orig_station", "walk_distance"]]
       .rename(columns={
           "DEST_STATION_NAME": "(BINARY_Only)_journey_dest_station",
           "EXIT_TM": "(BINARY_Only)_journey_exit_tm",
           "next_orig_station": "(BINARY_Only)_journey_next_orig_station",
           "walk_distance": "(BINARY_Only)_journey_walk_to_next_distance"
       })
)

binary_journey_info = binary_journey_first.merge(
    binary_journey_last,
    on=["CRD_NUM", "binary_full_journey_seq"],
    how="inner"
)

df3 = df3.merge(
    binary_journey_info,
    on=["CRD_NUM", "binary_full_journey_seq"],
    how="left"
)

print("\n=== Binary-only journey block ===")
print("df3 shape after binary journey merge:", df3.shape)
print("Duplicate RIDE_ID_NUM in df3:", df3["RIDE_ID_NUM"].duplicated().sum())
print("Binary journey seq nulls:", df3["binary_full_journey_seq"].isna().sum())


#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
    - Allowance = Walking speed * walking distance



Describe.

In [ ]:
TEMPORAL_SPEC = "strict"

In [ ]:
'''# Calculate time gap mins

# Shift next entry time within the same commuter group
df3['next_ENTRY_TM'] = df3.groupby('CRD_NUM')['ENTRY_TM'].shift(-1)
df3["next_TRNSPT_MODE_CD"] = df3.groupby("CRD_NUM")["TRNSPT_MODE_CD"].shift(-1)'''

In [ ]:
df3["mode_pair"] = np.select(
    [
        (df3["TRNSPT_MODE_CD"] == 1) & (df3["next_TRNSPT_MODE_CD"] == 1),
        (df3["TRNSPT_MODE_CD"] == 2) & (df3["next_TRNSPT_MODE_CD"] == 1),
        (df3["TRNSPT_MODE_CD"] == 1) & (df3["next_TRNSPT_MODE_CD"] == 2),
        (df3["TRNSPT_MODE_CD"] == 2) & (df3["next_TRNSPT_MODE_CD"] == 2)
    ],
    [
        "bus_bus",
        "train_bus",
        "bus_train",
        "train_train"
    ],
    default="other" # 'other' happens when the next_TRNSPT_MODE_CD does not exist (last stage)
)

In [ ]:
morning_peak = (
    ((df3["next_ENTRY_TM"].dt.hour == 6) & (df3["next_ENTRY_TM"].dt.minute >= 30)) |
    (df3["next_ENTRY_TM"].dt.hour.isin([7, 8]))
)

evening_peak = df3["next_ENTRY_TM"].dt.hour.isin([17, 18, 19])

df3["is_peak"] = morning_peak | evening_peak

df3["peak_period"] = np.select(
    [morning_peak, evening_peak],
    ["morning_peak", "evening_peak"],
    default="off_peak"
)

In [ ]:
df3["time_gap_mins"] = (
    df3["next_ENTRY_TM"] - df3["EXIT_TM"]
).dt.total_seconds() / 60

df3["predicted_walking_time_mins"] = (
    df3["walk_distance"] / df3["walking_speed_ms"]
) / 60

df3["waiting_time_plus_extra"] = (
    df3["time_gap_mins"] - df3["predicted_walking_time_mins"]
)

In [ ]:
# residual waiting allowance
# Used ONLY for bus_bus and train_bus

if TEMPORAL_SPEC == "strict":
    df3["waiting_time_allowed"] = np.select(
        [
            (df3["mode_pair"] == "bus_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (~df3["is_peak"]),
        ],
        [
            8,   # bus_bus peak
            12,  # bus_bus off_peak
            8,   # train_bus peak
            12,  # train_bus off_peak
        ],
        default=np.nan
    )

elif TEMPORAL_SPEC == "baseline":
    df3["waiting_time_allowed"] = np.select(
        [
            (df3["mode_pair"] == "bus_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (~df3["is_peak"]),
        ],
        [
            10,  # bus_bus peak
            15,  # bus_bus off_peak
            10,  # train_bus peak
            15,  # train_bus off_peak
        ],
        default=np.nan
    )

elif TEMPORAL_SPEC == "lenient":
    df3["waiting_time_allowed"] = np.select(
        [
            (df3["mode_pair"] == "bus_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (~df3["is_peak"]),
        ],
        [
            12,  # bus_bus peak
            18,  # bus_bus off_peak
            12,  # train_bus peak
            18,  # train_bus off_peak
        ],
        default=np.nan
    )

else:
    raise ValueError("TEMPORAL_SPEC must be 'strict', 'baseline', or 'lenient'")

In [ ]:
df3["extras_allowed"] = 0
df3["Total_allowance"] = df3["waiting_time_allowed"] + df3["extras_allowed"]

In [ ]:
# separate train-entry buffer rule
# Used ONLY for bus_train and train_train

TRAIN_ENTRY_BUFFER_MINS = 5

df3["before_train_entry_buffer_mins"] = np.where(
    df3["mode_pair"].isin(["bus_train", "train_train"]),
    TRAIN_ENTRY_BUFFER_MINS,
    np.nan
)

df3["train_entry_total_allowed_mins"] = (
    df3["predicted_walking_time_mins"] + df3["before_train_entry_buffer_mins"]
)


In [ ]:
# Temporal flag
# bus_bus / train_bus: residual waiting rule
# bus_train / train_train: walking + fixed buffer rule

bus_entries = df3["mode_pair"].isin(["bus_bus", "train_bus"])
train_entries = df3["mode_pair"].isin(["bus_train", "train_train"])

df3["temporal_flag_reason"] = np.select(
    [
        df3["time_gap_mins"].isna(),
        df3["predicted_walking_time_mins"].isna(),

        # bus_bus / train_bus only
        bus_entries & df3["waiting_time_plus_extra"].isna(),
        bus_entries & df3["waiting_time_allowed"].isna(),
        bus_entries & (df3["waiting_time_plus_extra"] > df3["Total_allowance"]),

        # bus_train / train_train only
        train_entries & df3["train_entry_total_allowed_mins"].isna(),
        train_entries & (df3["time_gap_mins"] > df3["train_entry_total_allowed_mins"]),

        # other
        df3["mode_pair"] == "other",
    ],
    [
        "null_time_gap",
        "null_predicted_walking_time",

        "null_waiting_time_plus_extra",
        "unknown_mode_pair",
        "exceeds_total_allowance",

        "null_train_entry_total_allowed_mins",
        "exceeds_train_entry_buffer",

        "unknown_mode_pair",
    ],
    default="within_total_allowance"
)

df3["exceeds_time_allowance"] = (
    df3["temporal_flag_reason"] != "within_total_allowance"
)

In [ ]:
print(df3["exceeds_time_allowance"].value_counts(dropna=False))
print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")
print(df3["temporal_flag_reason"].value_counts(dropna=False))

print("\nMode pair:")
print(df3["mode_pair"].value_counts(dropna=False))

print("\nPeak period:")
print(df3["peak_period"].value_counts(dropna=False))

print("\nWaiting time allowed:")
print(df3["waiting_time_allowed"].value_counts(dropna=False))

print("\nBefore train entry buffer mins:")
print(df3["before_train_entry_buffer_mins"].value_counts(dropna=False))

print("\nTrain entry total allowed mins:")
print(df3["train_entry_total_allowed_mins"].value_counts(dropna=False))

In [ ]:
df3["temporal_full_journey_seq"] = (
    df3.groupby("CRD_NUM")["exceeds_time_allowance"]
       .shift(fill_value=False)
       .groupby(df3["CRD_NUM"])
       .cumsum()
)

temporal_journey_first = (
    df3.drop_duplicates(subset=["CRD_NUM", "temporal_full_journey_seq"], keep="first")
       [["CRD_NUM", "temporal_full_journey_seq", "ORIG_STATION_NAME", "ENTRY_TM"]]
       .rename(columns={
           "ORIG_STATION_NAME": "(TEMPORAL_Only)_journey_orig_station",
           "ENTRY_TM": "(TEMPORAL_Only)_journey_entry_tm"
       })
)

temporal_journey_last = (
    df3.drop_duplicates(subset=["CRD_NUM", "temporal_full_journey_seq"], keep="last")
       [["CRD_NUM", "temporal_full_journey_seq", "DEST_STATION_NAME", "EXIT_TM"]]
       .rename(columns={
           "DEST_STATION_NAME": "(TEMPORAL_Only)_journey_dest_station",
           "EXIT_TM": "(TEMPORAL_Only)_journey_exit_tm"
       })
)

temporal_journey_info = temporal_journey_first.merge(
    temporal_journey_last,
    on=["CRD_NUM", "temporal_full_journey_seq"],
    how="inner"
)

df3 = df3.merge(
    temporal_journey_info,
    on=["CRD_NUM", "temporal_full_journey_seq"],
    how="left"
)

* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

## Using Binary and Temporal
- apply temporal criteria only after filtering transfers from binary

In [ ]:
# Binary + Temporal combined
df3["BT_flag"] = (
    df3["binary_flag"] |
    df3["exceeds_time_allowance"]
)

df3["BT_flag_reason"] = ""

# binary
df3.loc[df3["binary_flag"], "BT_flag_reason"] += (
    df3["binary_flag_reason"].fillna("") + " | "
)

# temporal
df3.loc[df3["exceeds_time_allowance"], "BT_flag_reason"] += (
    df3["temporal_flag_reason"].fillna("") + " | "
)

df3["BT_flag_reason"] = (
    df3["BT_flag_reason"]
    .str.strip(" | ")
)

df3.loc[df3["BT_flag_reason"] == "", "BT_flag_reason"] = "candidate_transfer"

print("\n=== Binary + Temporal ===")
print(df3["BT_flag"].value_counts(dropna=False))
print(df3["BT_flag_reason"].value_counts(dropna=False).head(20))

# optional safety check: make sure merge keys are unique in df3
print("\nDuplicate merge keys in df3:")
print(df3.duplicated(["CRD_NUM", "ENTRY_TM", "EXIT_TM"]).sum())

## Hard geographic threshold (consider)

In [ ]:
SPATIAL_THRESHOLD_M = 1200

df3['spatial_flag_reason'] = np.select(
    [
        df3['walk_distance'].isna(),
        df3['walk_distance'] > SPATIAL_THRESHOLD_M
    ],
    [
        'null_walk_distance',
        'exceeds_distance_threshold'
    ],
    default='within_distance_threshold'
)

df3['exceeds_walking_dist_threshold'] = (
    df3['spatial_flag_reason'] != 'within_distance_threshold'
)

print("\n=== Spatial distance rule ===")
print(df3["exceeds_walking_dist_threshold"].value_counts(dropna=False))
print(df3["spatial_flag_reason"].value_counts(dropna=False))

In [ ]:
df3["spatial_full_journey_seq"] = (
    df3.groupby("CRD_NUM")["exceeds_walking_dist_threshold"]
       .shift(fill_value=False)
       .groupby(df3["CRD_NUM"])
       .cumsum()
)

spatial_journey_first = (
    df3.drop_duplicates(subset=["CRD_NUM", "spatial_full_journey_seq"], keep="first")
       [["CRD_NUM", "spatial_full_journey_seq", "ORIG_STATION_NAME"]]
       .rename(columns={
           "ORIG_STATION_NAME": "(SPATIAL_Only)_journey_orig_station"
       })
)

spatial_journey_last = (
    df3.drop_duplicates(subset=["CRD_NUM", "spatial_full_journey_seq"], keep="last")
       [["CRD_NUM", "spatial_full_journey_seq", "DEST_STATION_NAME"]]
       .rename(columns={
           "DEST_STATION_NAME": "(SPATIAL_Only)_journey_dest_station"
       })
)

spatial_journey_info = spatial_journey_first.merge(
    spatial_journey_last,
    on=["CRD_NUM", "spatial_full_journey_seq"],
    how="inner"
)

df3 = df3.merge(
    spatial_journey_info,
    on=["CRD_NUM", "spatial_full_journey_seq"],
    how="left"
)


In [ ]:
df3["(SPATIAL_Only)_short_round_trip_flag"] = (
    (df3["(SPATIAL_Only)_journey_orig_station"] == df3["(SPATIAL_Only)_journey_dest_station"]) &
    df3["(SPATIAL_Only)_journey_orig_station"].notna() &
    df3["(SPATIAL_Only)_journey_dest_station"].notna()
)

df3["(SPATIAL_Only)_round_trip_reason"] = np.select(
    [
        df3["(SPATIAL_Only)_journey_orig_station"].isna() |
        df3["(SPATIAL_Only)_journey_dest_station"].isna(),
        df3["(SPATIAL_Only)_journey_orig_station"] == df3["(SPATIAL_Only)_journey_dest_station"]
    ],
    [
        "null_origin_or_destination",
        "same_origin_and_destination"
    ],
    default="different_origin_destination"
)

df3["spatial_flag"] = (
    df3["exceeds_walking_dist_threshold"] |
    df3["(SPATIAL_Only)_short_round_trip_flag"]
)

print(df3["(SPATIAL_Only)_short_round_trip_flag"].value_counts(dropna=False))
print(df3["(SPATIAL_Only)_round_trip_reason"].value_counts(dropna=False))
print(df3["spatial_flag"].value_counts(dropna=False))

### Combining all 3 criteria

In [ ]:
# -----------------------------------
# Binary + Temporal + Spatial combined
# -----------------------------------
df3["BTS_flag"] = (
    df3["binary_flag"] |
    df3["exceeds_time_allowance"] |
    df3["exceeds_walking_dist_threshold"] |
    df3["(SPATIAL_Only)_short_round_trip_flag"]
)

df3["BTS_flag_reason"] = ""

df3.loc[df3["binary_flag"], "BTS_flag_reason"] += (
    df3["binary_flag_reason"].fillna("") + " | "
)

df3.loc[df3["exceeds_time_allowance"], "BTS_flag_reason"] += (
    df3["temporal_flag_reason"].fillna("") + " | "
)

df3.loc[df3["exceeds_walking_dist_threshold"], "BTS_flag_reason"] += (
    df3["spatial_flag_reason"].fillna("") + " | "
)

df3.loc[df3["(SPATIAL_Only)_short_round_trip_flag"], "BTS_flag_reason"] += (
    df3["(SPATIAL_Only)_round_trip_reason"].fillna("") + " | "
)

df3["BTS_flag_reason"] = (
    df3["BTS_flag_reason"]
    .str.strip(" | ")
)

df3.loc[df3["BTS_flag_reason"] == "", "BTS_flag_reason"] = "candidate_transfer"

print("\n=== Provisional BTS (using SPATIAL_Only round trip) ===")
print(df3["BTS_flag"].value_counts(dropna=False))
print(df3["BTS_flag_reason"].value_counts(dropna=False).head(30))

In [ ]:
df3 = df3.sort_values(["CRD_NUM", "ENTRY_TM"]).reset_index(drop=True)

df3["BTS_journey_seq"] = (
    df3.groupby("CRD_NUM")["BTS_flag"]
       .shift(fill_value=False)
       .groupby(df3["CRD_NUM"])
       .cumsum()
)

# first row of each BTS journey
BTS_journey_first = (
    df3.drop_duplicates(subset=["CRD_NUM", "BTS_journey_seq"], keep="first")
       [[
           "CRD_NUM",
           "BTS_journey_seq",
           "ORIG_STATION_NAME",
           "ENTRY_TM"
       ]]
       .rename(columns={
           "ORIG_STATION_NAME": "BTS_journey_orig_station",
           "ENTRY_TM": "BTS_journey_entry_tm"
       })
)

# last row of each BTS journey
BTS_journey_last = (
    df3.drop_duplicates(subset=["CRD_NUM", "BTS_journey_seq"], keep="last")
       [[
           "CRD_NUM",
           "BTS_journey_seq",
           "DEST_STATION_NAME",
           "EXIT_TM"
       ]]
       .rename(columns={
           "DEST_STATION_NAME": "BTS_journey_dest_station",
           "EXIT_TM": "BTS_journey_exit_tm"
       })
)

BTS_journey_info = BTS_journey_first.merge(
    BTS_journey_last,
    on=["CRD_NUM", "BTS_journey_seq"],
    how="inner"
)

df3 = df3.merge(
    BTS_journey_info,
    on=["CRD_NUM", "BTS_journey_seq"],
    how="left"
)

print("\n=== BTS journey sequence ===")
print("Null BTS_journey_seq:", df3["BTS_journey_seq"].isna().sum())
print("Duplicate RIDE_ID_NUM in df3:", df3["RIDE_ID_NUM"].duplicated().sum())


In [ ]:
df3["(BTS_Only)_short_round_trip_flag"] = (
    (df3["BTS_journey_orig_station"] == df3["BTS_journey_dest_station"]) &
    df3["BTS_journey_orig_station"].notna() &
    df3["BTS_journey_dest_station"].notna()
)

df3["(BTS_Only)_round_trip_reason"] = np.select(
    [
        df3["BTS_journey_orig_station"].isna() |
        df3["BTS_journey_dest_station"].isna(),

        df3["BTS_journey_orig_station"] == df3["BTS_journey_dest_station"]
    ],
    [
        "null_origin_or_destination",
        "same_origin_and_destination"
    ],
    default="different_origin_destination"
)

print("\n=== BTS-only round trip ===")
print(df3["(BTS_Only)_short_round_trip_flag"].value_counts(dropna=False))
print(df3["(BTS_Only)_round_trip_reason"].value_counts(dropna=False))

In [ ]:
df3["BTS_flag"] = (
    df3["binary_flag"] |
    df3["exceeds_time_allowance"] |
    df3["exceeds_walking_dist_threshold"] |
    df3["(BTS_Only)_short_round_trip_flag"]
)

df3["BTS_flag_reason"] = ""

# binary
df3.loc[df3["binary_flag"], "BTS_flag_reason"] += (
    df3["binary_flag_reason"].fillna("") + " | "
)

# temporal
df3.loc[df3["exceeds_time_allowance"], "BTS_flag_reason"] += (
    df3["temporal_flag_reason"].fillna("") + " | "
)

# spatial distance
df3.loc[df3["exceeds_walking_dist_threshold"], "BTS_flag_reason"] += (
    df3["spatial_flag_reason"].fillna("") + " | "
)

# final round trip: BTS-only
df3.loc[df3["(BTS_Only)_short_round_trip_flag"], "BTS_flag_reason"] += (
    df3["(BTS_Only)_round_trip_reason"].fillna("") + " | "
)

df3["BTS_flag_reason"] = (
    df3["BTS_flag_reason"]
    .str.strip(" | ")
)

df3.loc[df3["BTS_flag_reason"] == "", "BTS_flag_reason"] = "candidate_transfer"

print("\n=== Final BTS (with distance + BTS-only round trip) ===")
print(df3["BTS_flag"].value_counts(dropna=False))
print(df3["BTS_flag_reason"].value_counts(dropna=False).head(40))

In [ ]:
df3["BTS_flag_nodist"] = (
    df3["binary_flag"] |
    df3["exceeds_time_allowance"] |
    df3["(BTS_Only)_short_round_trip_flag"]
)

df3["BTS_flag_reason_nodist"] = ""

# binary
df3.loc[df3["binary_flag"], "BTS_flag_reason_nodist"] += (
    df3["binary_flag_reason"].fillna("") + " | "
)

# temporal
df3.loc[df3["exceeds_time_allowance"], "BTS_flag_reason_nodist"] += (
    df3["temporal_flag_reason"].fillna("") + " | "
)

# round trip only
df3.loc[df3["(BTS_Only)_short_round_trip_flag"], "BTS_flag_reason_nodist"] += (
    df3["(BTS_Only)_round_trip_reason"].fillna("") + " | "
)

df3["BTS_flag_reason_nodist"] = (
    df3["BTS_flag_reason_nodist"]
    .str.strip(" | ")
)

df3.loc[df3["BTS_flag_reason_nodist"] == "", "BTS_flag_reason_nodist"] = "candidate_transfer"

print("\n=== BTS no-distance ===")
print(df3["BTS_flag_nodist"].value_counts(dropna=False))
print(df3["BTS_flag_reason_nodist"].value_counts(dropna=False).head(40))

In [ ]:
df3["is_same_journey_final"] = ~df3["BTS_flag"]
df3["is_same_journey_final_nodist"] = ~df3["BTS_flag_nodist"]

print("\n=== Final same-journey outputs ===")
print(df3["is_same_journey_final"].value_counts(dropna=False))
print(df3["is_same_journey_final_nodist"].value_counts(dropna=False))
print("Duplicate RIDE_ID_NUM in df3:", df3["RIDE_ID_NUM"].duplicated().sum())

## Internal Validity Check with pt_jrny

In [ ]:
df5 = pd.read_pickle('../data/df5.pkl')
df5.info()

In [ ]:
df5['service_day'] = (df5['JRNY_START_TM'] - pd.Timedelta(hours=5)).dt.date

target_day = pd.Timestamp('2025-02-12').date()

df5 = df5[df5['service_day'] == target_day].reset_index(drop=True)

In [ ]:
# same filtering as df3
df5 = df5[df5['PATRON_CATG_ID_NUM'].isin([1, 3, 4])].reset_index(drop=True)

In [ ]:
# Merge on CRD_NUM + JRNY_ID_NUM to attach journey boundaries to each ride
df_val = df3.merge(
    df5[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM']],
    on=['CRD_NUM', 'JRNY_ID_NUM'],
    how='inner'
)

# Keep only rides that fall within their journey's time window
df_val = df_val[
    (df_val['ENTRY_TM'] >= df_val['JRNY_START_TM']) &
    (df_val['EXIT_TM'] <= df_val['JRNY_END_TM'])
].reset_index(drop=True)

print(f"Rows after merge + time filter: {len(df_val):,}")

In [ ]:
df_val = df_val.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)
# Shift JRNY_ID_NUM within each commuter to get the next ride's journey ID
df_val['next_JRNY_ID_NUM'] = df_val.groupby('CRD_NUM')['JRNY_ID_NUM'].shift(-1)

# true_transfer = True → same journey (actual transfer)
# true_transfer = False → different journey (actual new journey)
df_val['true_transfer'] = (df_val['JRNY_ID_NUM'] == df_val['next_JRNY_ID_NUM'])

# Drop last stage of each commuter — no next ride to compare against
df_val = df_val[df_val['next_JRNY_ID_NUM'].notna()].reset_index(drop=True)

print(df_val['true_transfer'].value_counts())

In [ ]:
print("\nMode pair:")
print(df_val['mode_pair'].value_counts(dropna=False))

print("\nPeak period:")
print(df_val['peak_period'].value_counts(dropna=False))

print("\nWaiting time allowed:")
print(df_val['waiting_time_allowed'].value_counts(dropna=False))

print("\nMode pair x peak period:")
print(pd.crosstab(df_val['mode_pair'], df_val['peak_period']))

print("\nMode pair x waiting time allowed:")
print(pd.crosstab(df_val['mode_pair'], df_val['waiting_time_allowed']))

In [ ]:
# helper function
def print_metrics(df, pred_flag_col, label):
    """
    pred_flag_col: True = classifier says NEW JOURNEY, False = classifier says TRANSFER
    true_transfer: True = actual transfer, False = actual new journey
    """
    actual_transfer = df['true_transfer']
    pred_transfer = ~df[pred_flag_col]      # flip: flag=True means new journey, so ~flag = predicted transfer

    tp = (actual_transfer & pred_transfer).sum()     # actual transfer, predicted transfer
    tn = (~actual_transfer & ~pred_transfer).sum()   # actual new journey, predicted new journey
    fp = (~actual_transfer & pred_transfer).sum()    # actual new journey, predicted transfer (merge error)
    fn = (actual_transfer & ~pred_transfer).sum()    # actual transfer, predicted new journey (split error)

    total_actual_transfers = actual_transfer.sum()

    print(f"\n=== {label} ===")
    print(f"TP: {tp:,}  TN: {tn:,}  FP: {fp:,}  FN: {fn:,}")
    print(f"Split rate  (FN / actual transfers): {fn / total_actual_transfers:.4f}")
    print(f"Merge rate  (FP / actual transfers): {fp / total_actual_transfers:.4f}")
    print(f"Sensitivity (TP / (TP+FN)):          {tp / (tp + fn):.4f}")
    print(f"Specificity (TN / (TN+FP)):          {tn / (tn + fp):.4f}")
    print(f"Accuracy    ((TP+TN) / total):       {(tp + tn) / len(df):.4f}")

    print(pd.crosstab(
        actual_transfer,
        pred_transfer,
        rownames=['Actual transfer'],
        colnames=[f'Predicted transfer ({label})']
    ))

In [ ]:
print_metrics(df_val, 'binary_flag', 'Binary Criteria')

In [ ]:
print_metrics(df_val, 'exceeds_time_allowance', 'Temporal Criteria')

In [ ]:
print_metrics(df_val, 'exceeds_walking_dist_threshold', 'Spatial Criteria (Distance Threshold)')

In [ ]:
print_metrics(df_val, '(BTS_Only)_short_round_trip_flag', 'Spatial Criteria (Round Trip)')

In [ ]:
print_metrics(df_val, 'spatial_flag', 'Spatial Criteria (Combined)')

In [ ]:
print_metrics(df_val, 'BT_flag', 'Binary + Temporal')

In [ ]:
print_metrics(
    df_val,
    'BTS_flag',
    'Binary + Temporal + Spatial (With Distance Threshold)'
)

In [ ]:
print_metrics(
    df_val,
    'BTS_flag_nodist',
    'Binary + Temporal + Spatial (Round Trip Only)'
)

In [ ]:
temporal_errors = df_val[
    df_val['exceeds_time_allowance'] != (~df_val['true_transfer'])
].copy()

print(f"\nRows where temporal criterion disagrees with benchmark: {len(temporal_errors):,}")

print("\nTemporal errors by mode pair:")
print(temporal_errors['mode_pair'].value_counts(dropna=False))

print("\nTemporal errors by peak period:")
print(temporal_errors['peak_period'].value_counts(dropna=False))

print("\nTemporal errors by waiting time allowed:")
print(temporal_errors['waiting_time_allowed'].value_counts(dropna=False))

print("\nTemporal errors by reason:")
print(temporal_errors['temporal_flag_reason'].value_counts(dropna=False))

# journey dataset that imposes all criterias

In [ ]:
df3 = df3.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)

# define final journey sequence using BTS_flag
df3['BTS_journey_seq'] = (
    df3.groupby('CRD_NUM')['BTS_flag']
       .shift(fill_value=False)
       .groupby(df3['CRD_NUM'])
       .cumsum()
)

# first row of each journey
journey_first = (
    df3.drop_duplicates(subset=['CRD_NUM', 'BTS_journey_seq'], keep='first')
       [[
           'CRD_NUM',
           'BTS_journey_seq',
           'ORIG_STATION_NAME',
           'ORIG_LOC_ID_NUM',
           'ORIG_MRK_ID_NUM',
           'ORIG_LATITUDE',
           'ORIG_LONGITUDE',
           'ENTRY_TM',
           'TRNSPT_MODE_CD',
           'PATRON_CATG_ID_NUM'
       ]]
       .rename(columns={
           'ORIG_STATION_NAME': 'journey_orig_station',
           'ORIG_LOC_ID_NUM': 'journey_orig_loc_id',
           'ORIG_MRK_ID_NUM': 'journey_orig_mrk_id',
           'ORIG_LATITUDE': 'journey_orig_latitude',
           'ORIG_LONGITUDE': 'journey_orig_longitude',
           'ENTRY_TM': 'journey_entry_tm',
           'TRNSPT_MODE_CD': 'journey_first_mode',
           'PATRON_CATG_ID_NUM': 'journey_patron_catg'
       })
)

# last row of each journey
journey_last = (
    df3.drop_duplicates(subset=['CRD_NUM', 'BTS_journey_seq'], keep='last')
       [[
           'CRD_NUM',
           'BTS_journey_seq',
           'DEST_STATION_NAME',
           'DEST_LOC_ID_NUM',
           'DEST_MRK_ID_NUM',
           'DEST_LATITUDE',
           'DEST_LONGITUDE',
           'EXIT_TM',
           'TRNSPT_MODE_CD',
           'walk_distance',
           'next_orig_station',
           'BTS_flag_reason'
       ]]
       .rename(columns={
           'DEST_STATION_NAME': 'journey_dest_station',
           'DEST_LOC_ID_NUM': 'journey_dest_loc_id',
           'DEST_MRK_ID_NUM': 'journey_dest_mrk_id',
           'DEST_LATITUDE': 'journey_dest_latitude',
           'DEST_LONGITUDE': 'journey_dest_longitude',
           'EXIT_TM': 'journey_exit_tm',
           'TRNSPT_MODE_CD': 'journey_last_mode',
           'walk_distance': 'walk_to_next_journey_distance',
           'next_orig_station': 'next_journey_orig_station',
           'BTS_flag_reason': 'journey_end_reason'
       })
)

# aggregates
journey_summary = (
    df3.groupby(['CRD_NUM', 'BTS_journey_seq'], as_index=False)
       .agg(
           num_stages=('ENTRY_TM', 'size'),
           total_ride_fare=('RIDE_FARE_AMT', 'sum'),
           total_ride_distance_km=('RIDE_DIST_KM_CNT', 'sum'),
           total_ride_time_mins=('RIDE_MIN_CNT', 'sum')
       )
)

# combine
df_journey = (
    journey_first
    .merge(journey_last, on=['CRD_NUM', 'BTS_journey_seq'], how='inner')
    .merge(journey_summary, on=['CRD_NUM', 'BTS_journey_seq'], how='left')
    .drop_duplicates(subset=['CRD_NUM', 'BTS_journey_seq'])
    .reset_index(drop=True)
)

# duration
df_journey['journey_duration_mins'] = (
    df_journey['journey_exit_tm'] - df_journey['journey_entry_tm']
).dt.total_seconds() / 60

print(df_journey.shape)
print(df_journey[['CRD_NUM', 'BTS_journey_seq']].duplicated().sum(), "duplicate journey keys")

In [ ]:
gc.collect()

In [ ]:
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

# LOAD SAVED PLANNING AREA JSON
with open('../../data/planning_areas_2019.json', 'r') as f:
    planning_area_json = json.load(f)

areas = pd.DataFrame(planning_area_json['SearchResults']).copy()
areas['geojson_parsed'] = areas['geojson'].apply(json.loads)
areas['geometry'] = areas['geojson_parsed'].apply(shape)

planning_area_gdf = gpd.GeoDataFrame(
    areas,
    geometry='geometry',
    crs='EPSG:4326'
)[['pln_area_n', 'geometry']].rename(columns={
    'pln_area_n': 'planning_area'
})

# HELPER FUNCTION TO ASSIGN PLANNING AREA
def get_planning_area(df, lat_col, lon_col):
    gdf_points = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326'
    )

    joined = gpd.sjoin(
        gdf_points,
        planning_area_gdf,
        how='left',
        predicate='within'
    )

    return joined['planning_area'].values

# ADD REGION COLUMNS TO df3
df3['orig_region'] = get_planning_area(df3, 'ORIG_LATITUDE', 'ORIG_LONGITUDE')
df3['dest_region'] = get_planning_area(df3, 'DEST_LATITUDE', 'DEST_LONGITUDE')

# HOUSE AREA = FIRST ORIGIN OBSERVED FOR EACH CRD_NUM
first_tap = (
    df3.sort_values(['CRD_NUM', 'ENTRY_TM'])
       .drop_duplicates(subset=['CRD_NUM'], keep='first')
       .copy()
)

first_tap['house_area'] = get_planning_area(
    first_tap,
    'ORIG_LATITUDE',
    'ORIG_LONGITUDE'
)

df3 = df3.merge(
    first_tap[['CRD_NUM', 'house_area']],
    on='CRD_NUM',
    how='left'
)

In [ ]:
# ADD REGION COLUMNS TO df_journey
df_journey['journey_orig_region'] = get_planning_area(
    df_journey,
    'journey_orig_latitude',
    'journey_orig_longitude'
)

df_journey['journey_dest_region'] = get_planning_area(
    df_journey,
    'journey_dest_latitude',
    'journey_dest_longitude'
)

# MERGE HOUSE AREA INTO df_journey
df_journey = df_journey.merge(
    first_tap[['CRD_NUM', 'house_area']],
    on='CRD_NUM',
    how='left'
)

In [ ]:
df3.columns

In [ ]:
print("df3 orig_region NaNs:", df3['orig_region'].isna().sum())
print("df3 dest_region NaNs:", df3['dest_region'].isna().sum())
print("df3 house_area NaNs:", df3['house_area'].isna().sum())


print("df_journey journey_orig_region NaNs:", df_journey['journey_orig_region'].isna().sum())
print("df_journey journey_dest_region NaNs:", df_journey['journey_dest_region'].isna().sum())
print("df_journey house_area NaNs:", df_journey['house_area'].isna().sum())

In [ ]:
# everyday we pickling
df3.to_pickle('../data/df3_strict_with_regions.pkl')
df_journey.to_pickle('../data/df_strict_journey_with_regions.pkl')

print('Saved df3_strict_with_regions.pkl')
print('Saved df_strict_journey_with_regions.pkl')

# Dont run this, just for checking

In [ ]:
'''rides_ours = df4[:300].copy()

rides_ours.to_csv('rides_ours.csv', index=False)

# keep only common commuters
common_crds = set(df_journey['CRD_NUM']).intersection(set(df5['CRD_NUM']))

ours = df_journey[df_journey['CRD_NUM'].isin(common_crds)].copy()
theirs = df5[df5['CRD_NUM'].isin(common_crds)].copy()

# sort so first 200 is meaningful
ours = ours.sort_values(['CRD_NUM', 'journey_entry_tm']).reset_index(drop=True)
theirs = theirs.sort_values(['CRD_NUM', 'JRNY_START_TM']).reset_index(drop=True)

# take first 200 rows
ours_200 = ours.head(200).copy()
theirs_200 = theirs.head(200).copy()

# export
ours_200.to_csv('ours_200.csv', index=False)
theirs_200.to_csv('theirs_200.csv', index=False)'''